# Fineturner ResNet pour une LiveNess Detection

Ce notebook est fait pour détailler le funeturning de __CelebA-Spoof__


## Import des bibliothèques

In [1]:

import io 
import zipfile
from pathlib import Path
from typing import Dict, List, Tuple, cast
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms

## Montage depuis le Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Chargement des poids de ResNet
Ici je personnalise la tête de ResNet et l'achitecture finale devient
2048 --> 256 --> 2 

In [3]:
def create_model(num_classes: int = 2, pretrained: bool = True) -> nn.Module:
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet18(weights=weights)
    
    # Geler tout le réseau de base si nécéssaire
    if pretrained :
        for param in model.parameters():
            param.requires_grad = False
    
    # Tête multi-couches avec Dropout
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    )
    
    return model

## Chargement du DataSet depuis des fichier zip

In [9]:
class ZipDataset(Dataset):

  def __init__(self, zip_path: str, split: str = 'train', transform=None):
    """Dataset PyTorch lisant directement depuis un fichier ZIP.

    Args:
        zip_path (str): Chemin vers le fichier zip.
        split (str): 'train' ou 'test'.
        transform (callable, optional): Transformations PyTorch à appliquer.
    """
    self.zip_path = zip_path
    self.split = split
    self.transform = transform

  
    # 'live' -> 0, 'spoof' -> 1 
    self.class_to_idx = {'live': 0, 'spoof': 1}

    # Pré-requis d'extensions valides
    valid_extensions = (
        '.jpg',
        '.jpeg',
        '.png',
        '.bmp',
        '.JPG',
        '.JPEG',
        '.PNG',
    )

    # Construction du préfixe exact attendu dans le zip
    # Ex: "Spoof_Live/CelebA_Spoof/train/"
    self.prefix = f'CelebA_Spoof/{split}/'

    self.image_paths = []
    self.labels = []
    
    self.zip_file = zipfile.ZipFile(self.zip_path, 'r')
    
    for file_path in self.zip_file.namelist():
      if file_path.startswith('__MACOSX') or not file_path.endswith(valid_extensions):
        continue

      parts = file_path.split('/')

      if self.split in parts:
        split_idx = parts.index(self.split)
        if split_idx + 1 < len(parts):
          class_name = parts[split_idx + 1]
          if class_name in self.class_to_idx:
            self.image_paths.append(file_path)
            self.labels.append(self.class_to_idx[class_name])
      
    

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self, idx):
    file_path = self.image_paths[idx]
    label = self.labels[idx]

    # Lecture de l'image binaire depuis l'archive sans extraction sur disque
    with zipfile.ZipFile(self.zip_path, 'r') as z:
      img_bytes = z.read(file_path)
      image = Image.open(io.BytesIO(img_bytes)).convert('RGB')

    # Application du prétraitement
    if self.transform is not None:
      image = self.transform(image)

    return image, label

## Transformations à appliquer au Images

In [5]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def build_transforms(train: bool = False, image_size: int = 224) -> transforms.Compose:
    if train:
        return transforms.Compose(
            [
                transforms.RandomResizedCrop(image_size),
                transforms.RandomHorizontalFlip(),
                # Variations de couleur
                transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
                # Effacer aléatoirement une petite zone 
                # et moins dépendant des artefacts locaux.
                transforms.RandomErasing(p=0.3, scale=(0.02, 0.1)),
            ]
        )
    return transforms.Compose(
        [
            transforms.Resize(int(image_size * 1.14)),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]
    )

## DataLoader

In [10]:
zip_file_path = "drive/MyDrive/Spoof_Live.zip"

train_dataset = ZipDataset(zip_path=zip_file_path , split="train" , transform=build_transforms(train=True))
test_dataset   = ZipDataset(zip_path=zip_file_path , split="test" , transform=build_transforms(train=False))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader   = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
model = create_model()

model = model.to(device)
criterion = nn.CrossEntropyLoss()

Device : cuda


In [ ]:
print(" Entraînement de la tête personnalisée ")

optimizer_head = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

for epoch in range(4): 
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer_head.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_head.step()
        
        running_loss += loss.item()
        
    print(f"Époque {epoch+1}/3 | Loss : {running_loss / len(train_loader):.4f}")


 Entraînement de la tête personnalisée 


In [ ]:

# ---------------------------------------------------------
# 4. ÉTAPE 2 : DÉGELER LES DERNIÈRES COUCHES ET FINETUNER
# ---------------------------------------------------------
print("\n--- PHASE 2 : Dégal du bloc 'layer4' et Fine-tuning ---")

# Dégeler le dernier bloc de convolutions
for param in model.layer4.parameters():
    param.requires_grad = True

# Optimiseur avec un Learning Rate plus petit pour le corps, plus grand pour la tête
optimizer_fine = torch.optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(),     'lr': 1e-3}
])

for epoch in range(5):  # Ré-entraînement global
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer_fine.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_fine.step()
        
        running_loss += loss.item()
        
    print(f"Époque {epoch+1}/5 | Loss : {running_loss / len(train_loader):.4f}")